In [9]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time


BASE_URL = "https://api.hsx.vn/mk/api/v1/market"
# Ví dụ dạng:
# BASE_URL = "https://www.hsx.vn/api/..."

VALUE_ENDPOINT = BASE_URL + "/foreign-trading-value-report"
VOLUME_ENDPOINT = BASE_URL + "/foreign-trading-volumn-report"


def clean_number(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x == "--":
        return None
    x = x.replace(",", "").replace("%", "").strip()
    return float(x)


def parse_hsx_value(response_json):
    result = {}

    for row in response_json.get("data", []):
        name = row.get("trading_type_name")

        if name == "Mua":
            result["foreign_buy_value_mil_vnd"] = clean_number(row.get("foreign_trading_value"))
            result["market_value_mil_vnd"] = clean_number(row.get("market_value"))
            result["foreign_buy_value_pct"] = clean_number(row.get("percent"))

        elif name == "Bán":
            result["foreign_sell_value_mil_vnd"] = clean_number(row.get("foreign_trading_value"))
            result["foreign_sell_value_pct"] = clean_number(row.get("percent"))

        elif name == "Mua - Bán":
            result["foreign_net_value_mil_vnd"] = clean_number(row.get("foreign_trading_value"))

    return result


def parse_hsx_volume(response_json):
    result = {}

    for row in response_json.get("data", []):
        name = row.get("trading_type_name")

        if name == "Mua":
            result["foreign_buy_volume_10shares"] = clean_number(row.get("foreign_trading_volumn"))
            result["market_volume_10shares"] = clean_number(row.get("market_volumn"))
            result["foreign_buy_volume_pct"] = clean_number(row.get("percent"))

        elif name == "Bán":
            result["foreign_sell_volume_10shares"] = clean_number(row.get("foreign_trading_volumn"))
            result["foreign_sell_volume_pct"] = clean_number(row.get("percent"))

        elif name == "Mua - Bán":
            result["foreign_net_volume_10shares"] = clean_number(row.get("foreign_trading_volumn"))

    return result


def daterange(start_date, end_date):
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        yield current.strftime("%Y-%m-%d")
        current += timedelta(days=1)


def fetch_json(url, date_str, market="Entire Market"):
    params = {
        "fromDate": date_str,
        "toDate": date_str,
        # phần này cần chỉnh theo payload thật của HSX nếu có thêm tham số
        # "code": market,
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json, text/plain, */*",
    }

    response = requests.get(url, params=params, headers=headers, timeout=20)
    response.raise_for_status()
    return response.json()


rows = []

for d in daterange("2019-02-01", "2019-09-01"):
    try:
        value_json = fetch_json(VALUE_ENDPOINT, d)
        volume_json = fetch_json(VOLUME_ENDPOINT, d)

        value_data = parse_hsx_value(value_json)
        volume_data = parse_hsx_volume(volume_json)

        row = {
            "date": d,
            "source": "HSX",
            **value_data,
            **volume_data,
        }

        # bỏ ngày không có giao dịch / không có dữ liệu
        has_data = (
            row.get("foreign_buy_value_mil_vnd") is not None or
            row.get("foreign_buy_volume_10shares") is not None
        )

        if has_data:
            rows.append(row)
            print("OK:", d)
        else:
            print("No data:", d)

        time.sleep(0.2)

    except Exception as e:
        print("ERROR:", d, e)
        rows.append({
            "date": d,
            "source": "HSX",
            "error": str(e)
        })


df = pd.DataFrame(rows)

# tạo thêm đơn vị tỷ VND
for col in [
    "foreign_buy_value_mil_vnd",
    "foreign_sell_value_mil_vnd",
    "foreign_net_value_mil_vnd",
    "market_value_mil_vnd"
]:
    if col in df.columns:
        df[col.replace("_mil_vnd", "_bil_vnd")] = df[col] / 1000

# tỷ lệ mua/bán ròng so với toàn thị trường
df["foreign_net_value_ratio"] = (
    df["foreign_net_value_mil_vnd"] / df["market_value_mil_vnd"]
)

df.to_csv("hsx_foreign_trading_2013_2026.csv", index=False, encoding="utf-8-sig")

print(df.head())
print("Saved:", "hsx_foreign_trading_2013_2026.csv")

OK: 2019-02-01
No data: 2019-02-02


KeyboardInterrupt: 

In [13]:
from pathlib import Path

def find_project_root(start=Path.cwd(), marker_files=('requirements.txt','LEAF_QUICKSTART.md')):
    p = Path(start).resolve()
    while True:
        if any((p / m).exists() for m in marker_files):
            return p
        if p.parent == p:
            return Path.cwd().resolve()
        p = p.parent

try:
    ROOT_DIR = Path(__file__).resolve().parent.parent.parent
except NameError:
    ROOT_DIR = find_project_root(Path.cwd())

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time


BASE_URL = "https://api.hsx.vn/mk/api/v1/market"

VALUE_ENDPOINT = BASE_URL + "/foreign-trading-value-report"
VOLUME_ENDPOINT = BASE_URL + "/foreign-trading-volumn-report"


def clean_number(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "" or x == "--":
        return None
    x = x.replace(",", "").replace("%", "").strip()
    return float(x)


def parse_hsx_value(response_json):
    """
    HSX Trading value đơn vị: mil VND.
    Output đổi sang tỷ VND để giống form CafeF.
    """
    result = {
        "GT Mua": None,
        "GT Bán": None,
        "GTGD ròng": None
    }

    for row in response_json.get("data", []):
        name = row.get("trading_type_name")
        value_mil = clean_number(row.get("foreign_trading_value"))

        if value_mil is None:
            continue

        value_bil = value_mil / 1000

        if name == "Mua":
            result["GT Mua"] = value_bil

        elif name == "Bán":
            result["GT Bán"] = value_bil

        elif name == "Mua - Bán":
            result["GTGD ròng"] = value_bil

    return result


def parse_hsx_volume(response_json):
    """
    HSX Trading vol. đơn vị: 10 shares.
    Output nhân 10 để ra số cổ phiếu.
    """
    result = {
        "KL Mua": None,
        "KL Bán": None,
        "KLGD ròng": None
    }

    for row in response_json.get("data", []):
        name = row.get("trading_type_name")
        volume_10shares = clean_number(row.get("foreign_trading_volumn"))

        if volume_10shares is None:
            continue

        volume_shares = volume_10shares * 10

        if name == "Mua":
            result["KL Mua"] = volume_shares

        elif name == "Bán":
            result["KL Bán"] = volume_shares

        elif name == "Mua - Bán":
            result["KLGD ròng"] = volume_shares

    return result


def daterange(start_date, end_date):
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        yield current.strftime("%Y-%m-%d")
        current += timedelta(days=1)


def fetch_json(url, date_str):
    params = {
        "fromDate": date_str,
        "toDate": date_str,
        "securitiesType": ""
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json, text/plain, */*",
    }

    response = requests.get(url, params=params, headers=headers, timeout=20)
    response.raise_for_status()
    return response.json()


rows = []

for d in daterange("2013-01-01", "2026-0-01"):
    try:
        value_json = fetch_json(VALUE_ENDPOINT, d)
        volume_json = fetch_json(VOLUME_ENDPOINT, d)

        value_data = parse_hsx_value(value_json)
        volume_data = parse_hsx_volume(volume_json)

        row = {
            "Ngày": d,
            **volume_data,
            **value_data
        }

        has_data = (
            row.get("KLGD ròng") is not None or
            row.get("GTGD ròng") is not None or
            row.get("KL Mua") is not None or
            row.get("GT Mua") is not None
        )

        if has_data:
            rows.append(row)
            print("OK:", d)
        else:
            print("No data:", d)

        time.sleep(0.2)

    except Exception as e:
        print("ERROR:", d, e)


# =========================
# Xuất đúng form bạn cần
# =========================

columns = [
    "Ngày",
    "KLGD ròng",
    "GTGD ròng",
    "KL Mua",
    "GT Mua",
    "KL Bán",
    "GT Bán"
]

df = pd.DataFrame(rows, columns=columns)

# Đổi ngày sang dd/mm/yyyy
df["Ngày"] = pd.to_datetime(df["Ngày"]).dt.strftime("%d/%m/%Y")

# Format giá trị giao dịch: tỷ VND
value_cols = ["GTGD ròng", "GT Mua", "GT Bán"]
for col in value_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

# Format khối lượng: cổ phiếu
volume_cols = ["KLGD ròng", "KL Mua", "KL Bán"]
for col in volume_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").round(0).astype("Int64")

df.to_excel("hsx_foreign_trading_form.xlsx", index=False)
df.to_csv("hsx_foreign_trading_form.csv", index=False, encoding="utf-8-sig")

print(df.head())
print("Saved: hsx_foreign_trading_form.xlsx")
print("Saved: hsx_foreign_trading_form.csv")

OK: 2013-01-02
OK: 2013-01-03
OK: 2013-01-04
OK: 2013-01-07
OK: 2013-01-08
OK: 2013-01-09
OK: 2013-01-10
OK: 2013-01-11
OK: 2013-01-14
OK: 2013-01-15
OK: 2013-01-16
OK: 2013-01-17
OK: 2013-01-18
OK: 2013-01-21
OK: 2013-01-22
OK: 2013-01-23
OK: 2013-01-24
OK: 2013-01-25
OK: 2013-01-28
OK: 2013-01-29
OK: 2013-01-30
OK: 2013-01-31
OK: 2013-02-01
OK: 2013-02-04
OK: 2013-02-05
OK: 2013-02-06
OK: 2013-02-07
OK: 2013-02-08
OK: 2013-02-18
OK: 2013-02-19
OK: 2013-02-20
OK: 2013-02-21
OK: 2013-02-22
OK: 2013-02-25
OK: 2013-02-26
OK: 2013-02-27
OK: 2013-02-28
OK: 2013-03-01
OK: 2013-03-04
OK: 2013-03-05
OK: 2013-03-06
OK: 2013-03-07
OK: 2013-03-08
OK: 2013-03-11
OK: 2013-03-12
OK: 2013-03-13
OK: 2013-03-14
OK: 2013-03-15
OK: 2013-03-18
OK: 2013-03-19
OK: 2013-03-20
OK: 2013-03-21
OK: 2013-03-22
OK: 2013-03-25
OK: 2013-03-26
OK: 2013-03-27
OK: 2013-03-28
OK: 2013-03-29
OK: 2013-04-01
OK: 2013-04-02
OK: 2013-04-03
OK: 2013-04-04
OK: 2013-04-05
OK: 2013-04-08
OK: 2013-04-09
OK: 2013-04-10
OK: 2013-0

KeyboardInterrupt: 

In [15]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd
from datetime import datetime, timedelta
import time
from pathlib import Path


# =========================
# CONFIG
# =========================

BASE_URL = "https://api.hsx.vn/mk/api/v1/market"

VALUE_ENDPOINT = BASE_URL + "/foreign-trading-value-report"
VOLUME_ENDPOINT = BASE_URL + "/foreign-trading-volumn-report"

# Định vị thư mục gốc
def find_project_root(start=Path.cwd(), marker_files=('requirements.txt','LEAF_QUICKSTART.md')):
    p = Path(start).resolve()
    while True:
        if any((p / m).exists() for m in marker_files):
            return p
        if p.parent == p:
            return Path.cwd().resolve()
        p = p.parent

ROOT_DIR = find_project_root()

# Update đường dẫn file
EXISTING_CSV = ROOT_DIR / "data" / "VNINDEX_Merged_2013_2026.csv"
OUTPUT_CSV = ROOT_DIR / "data" / "VNINDEX_Merged_2013_2026_filled.csv"
MISSING_ONLY_CSV = ROOT_DIR / "data" / "VNINDEX_Missing_Filled_From_HSX.csv"

START_DATE = "2013-01-01"
END_DATE = "2026-01-01"

columns = [
    "Ngày",
    "KLGD ròng",
    "GTGD ròng",
    "KL Mua",
    "GT Mua",
    "KL Bán",
    "GT Bán"
]


# =========================
# SESSION
# =========================

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[500, 502, 503, 504]
)

adapter = HTTPAdapter(max_retries=retries)
session.mount("http://", adapter)
session.mount("https://", adapter)


# =========================
# UTILS
# =========================

def clean_number(x):
    if x is None:
        return None

    x = str(x).strip()

    if x == "" or x == "--":
        return None

    x = x.replace(",", "").replace("%", "").strip()

    try:
        return float(x)
    except ValueError:
        return None


def trading_daterange(start_date, end_date):
    """
    Chỉ tạo ngày thứ 2 - thứ 6.
    Ngày nghỉ lễ vẫn có thể xuất hiện, nhưng API sẽ trả no data và mình bỏ qua.
    """
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        if current.weekday() < 5:
            yield current.strftime("%Y-%m-%d")

        current += timedelta(days=1)


def normalize_date_series(s):
    """
    File CSV của bạn có thể là 2/1/2013 hoặc 02/01/2013.
    dayfirst=True để hiểu đúng format Việt Nam.
    """
    return pd.to_datetime(s, dayfirst=True, errors="coerce")


def fetch_json(url, date_str):
    params = {
        "fromDate": date_str,
        "toDate": date_str,
        "securitiesType": ""
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json, text/plain, */*",
        "Connection": "keep-alive"
    }

    response = session.get(
        url,
        params=params,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()
    return response.json()


# =========================
# PARSE HSX VALUE
# =========================

def parse_hsx_value(response_json):
    """
    HSX Trading value đơn vị: mil VND.
    Đổi sang tỷ VND để giống file CSV hiện có.
    """

    result = {
        "GT Mua": None,
        "GT Bán": None,
        "GTGD ròng": None
    }

    if not response_json:
        return result

    for row in response_json.get("data", []):
        name = row.get("trading_type_name")
        value_mil = clean_number(row.get("foreign_trading_value"))

        if value_mil is None:
            continue

        value_bil = value_mil / 1000

        if name == "Mua":
            result["GT Mua"] = value_bil

        elif name == "Bán":
            result["GT Bán"] = value_bil

        elif name == "Mua - Bán":
            result["GTGD ròng"] = value_bil

    return result


# =========================
# PARSE HSX VOLUME
# =========================

def parse_hsx_volume(response_json):
    """
    HSX Trading vol. đơn vị: 10 shares.
    Nhân 10 để ra số cổ phiếu thực tế, giống CafeF / file CSV hiện có.
    """

    result = {
        "KL Mua": None,
        "KL Bán": None,
        "KLGD ròng": None
    }

    if not response_json:
        return result

    for row in response_json.get("data", []):
        name = row.get("trading_type_name")
        volume_10shares = clean_number(row.get("foreign_trading_volumn"))

        if volume_10shares is None:
            continue

        volume_shares = volume_10shares * 10

        if name == "Mua":
            result["KL Mua"] = volume_shares

        elif name == "Bán":
            result["KL Bán"] = volume_shares

        elif name == "Mua - Bán":
            result["KLGD ròng"] = volume_shares

    return result


# =========================
# LOAD EXISTING CSV
# =========================

existing_df = pd.read_csv(EXISTING_CSV)

# Giữ đúng các cột cần thiết
existing_df = existing_df[columns].copy()

# Parse ngày
existing_df["date_key"] = normalize_date_series(existing_df["Ngày"])

# Bỏ dòng lỗi ngày nếu có
existing_df = existing_df.dropna(subset=["date_key"])

# Chuẩn hóa key ngày dạng yyyy-mm-dd
existing_df["date_key_str"] = existing_df["date_key"].dt.strftime("%Y-%m-%d")

existing_dates = set(existing_df["date_key_str"])


# =========================
# FIND MISSING DATES
# =========================

expected_dates = list(trading_daterange(START_DATE, END_DATE))

missing_dates = [
    d for d in expected_dates
    if d not in existing_dates
]

print("Số ngày kỳ vọng (bỏ T7, CN):", len(expected_dates))
print("Số ngày đã có trong CSV:", len(existing_dates))
print("Số ngày còn thiếu cần crawl:", len(missing_dates))

if missing_dates:
    print("Ví dụ ngày thiếu đầu tiên:")
    print(missing_dates[:20])


# =========================
# CRAWL ONLY MISSING DATES
# =========================

new_rows = []
error_rows = []

for d in missing_dates:
    try:
        value_json = fetch_json(VALUE_ENDPOINT, d)
        volume_json = fetch_json(VOLUME_ENDPOINT, d)

        value_data = parse_hsx_value(value_json)
        volume_data = parse_hsx_volume(volume_json)

        row = {
            "Ngày": d,
            **volume_data,
            **value_data
        }

        has_data = any(
            row.get(col) is not None
            for col in [
                "KLGD ròng",
                "GTGD ròng",
                "KL Mua",
                "GT Mua",
                "KL Bán",
                "GT Bán"
            ]
        )

        if has_data:
            new_rows.append(row)
            print("OK:", d)
        else:
            # print("No data:", d)
            pass

        time.sleep(0.05)

    except Exception as e:
        print("ERROR:", d, e)
        error_rows.append({
            "Ngày": d,
            "error": str(e)
        })

        # Không nên sleep quá lâu
        time.sleep(0.5)


# =========================
# FORMAT NEW DATA
# =========================

new_df = pd.DataFrame(new_rows, columns=columns)

if not new_df.empty:
    new_df["Ngày"] = pd.to_datetime(new_df["Ngày"]).dt.strftime("%d/%m/%Y")

    # Giá trị giao dịch: tỷ VND
    for col in ["GTGD ròng", "GT Mua", "GT Bán"]:
        new_df[col] = pd.to_numeric(new_df[col], errors="coerce").round(2)

    # Khối lượng: cổ phiếu
    for col in ["KLGD ròng", "KL Mua", "KL Bán"]:
        new_df[col] = pd.to_numeric(new_df[col], errors="coerce").round(0).astype("Int64")

    new_df.to_csv(MISSING_ONLY_CSV, index=False, encoding="utf-8-sig")
    print("Saved missing-only file:", MISSING_ONLY_CSV)
else:
    print("Không crawl thêm được dòng mới nào.")


# =========================
# MERGE OLD + NEW
# =========================

# Chuẩn hóa lại existing về đúng format
existing_clean = existing_df[columns].copy()
existing_clean["Ngày"] = existing_df["date_key"].dt.strftime("%d/%m/%Y")

for col in ["GTGD ròng", "GT Mua", "GT Bán"]:
    existing_clean[col] = pd.to_numeric(existing_clean[col], errors="coerce").round(2)

for col in ["KLGD ròng", "KL Mua", "KL Bán"]:
    existing_clean[col] = pd.to_numeric(existing_clean[col], errors="coerce").round(0).astype("Int64")


final_df = pd.concat([existing_clean, new_df], ignore_index=True)

# Remove duplicate date nếu có
final_df["date_sort"] = pd.to_datetime(final_df["Ngày"], dayfirst=True, errors="coerce")
final_df = final_df.dropna(subset=["date_sort"])

final_df = (
    final_df
    .sort_values("date_sort")
    .drop_duplicates(subset=["date_sort"], keep="first")
)

final_df["Ngày"] = final_df["date_sort"].dt.strftime("%d/%m/%Y")
final_df = final_df[columns]

final_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved final merged file:", OUTPUT_CSV)
print("Final rows:", len(final_df))

if error_rows:
    error_df = pd.DataFrame(error_rows)
    error_log_path = ROOT_DIR / "data" / "hsx_missing_crawl_errors.csv"
    error_df.to_csv(error_log_path, index=False, encoding="utf-8-sig")
    print("Saved error log:", error_log_path)

Số ngày kỳ vọng (bỏ T7, CN): 3393
Số ngày đã có trong CSV: 2677
Số ngày còn thiếu cần crawl: 716
Ví dụ ngày thiếu đầu tiên:
['2013-01-01', '2013-02-11', '2013-02-12', '2013-02-13', '2013-02-14', '2013-02-15', '2013-04-19', '2013-04-29', '2013-04-30', '2013-05-01', '2013-09-02', '2014-01-01', '2014-01-28', '2014-01-29', '2014-01-30', '2014-01-31', '2014-02-03', '2014-02-04', '2014-02-05', '2014-04-09']
OK: 2014-10-28
OK: 2019-02-01
OK: 2019-02-11
OK: 2019-02-12
OK: 2019-02-13
OK: 2019-02-14
OK: 2019-02-15
OK: 2019-02-18
OK: 2019-02-19
OK: 2019-02-20
OK: 2019-02-21
OK: 2019-02-22
OK: 2019-02-25
OK: 2019-02-26
OK: 2019-02-27
OK: 2019-02-28
OK: 2019-03-01
OK: 2019-03-04
OK: 2019-03-05
OK: 2019-03-06
OK: 2019-03-07
OK: 2019-03-08
OK: 2019-03-11
OK: 2019-03-12
OK: 2019-03-13
OK: 2019-03-14
OK: 2019-03-15
OK: 2019-03-18
OK: 2019-03-19
OK: 2019-03-20
OK: 2019-03-21
OK: 2019-03-22
OK: 2019-03-25
OK: 2019-03-26
OK: 2019-03-27
OK: 2019-03-28
OK: 2019-03-29
OK: 2019-04-01
OK: 2019-04-02
OK: 2019-0